# FFT & Polynomial Multiplication

Multiplying two polynomials — equivalently, **convolving** two sequences — is naively O(n²): every coefficient of one meets every coefficient of the other. The **Fast Fourier Transform** drops that to **O(n log n)** by a change of representation: instead of multiplying coefficient-by-coefficient, evaluate both polynomials at the same `n` points, multiply the *values* pointwise (O(n)), then interpolate back. The magic is choosing the points to be the **complex n-th roots of unity**, which lets evaluation and interpolation share a divide-and-conquer structure (**divide & conquer**). This notebook builds the FFT from scratch, proves the convolution it computes equals the naive O(n²) one, then shows the production tool — `numpy.fft` — agreeing exactly. The angle, as ever: know the algorithm, reach for the C-level builtin.

**Contents**

1. **The problem** — convolution is O(n²)
2. **FFT** — recursive Cooley–Tukey + inverse, from scratch
3. **`numpy.fft`** — the production tool
4. **Applications** — big-integer multiply, counting
5. **NTT** — exact integer convolution over a prime field
6. **When to use + cheat-sheet**

## 1. The problem — convolution is O(n²)

A polynomial `A(x) = a₀ + a₁x + … + aₙ₋₁xⁿ⁻¹` is just its coefficient list `[a₀, …, aₙ₋₁]`. The product `C(x) = A(x)·B(x)` has coefficients given by the **convolution**

`c_k = Σ_{i+j=k} a_i · b_j`.

Done directly, that's a double loop — **O(n²)**. The same operation is everywhere: multiplying big integers (digits are coefficients), signal filtering, counting pair-sums, string matching with wildcards. The plan to beat O(n²):

| representation | multiply two polys | cost |
|---|---|:---:|
| **coefficients** | convolution (double loop) | O(n²) |
| **values** at n points | pointwise product | **O(n)** |

So if we could *cheaply* go coefficients → values (evaluate) and back (interpolate), we'd win. Naively, evaluating at n points is itself O(n²) — the FFT is the trick that makes the round trip O(n log n) by evaluating at the **roots of unity** (§2).

In [1]:
import random
random.seed(0)

def naive_conv(p, q):
    """Schoolbook convolution = polynomial multiply. O(len(p) * len(q))."""
    out = [0] * (len(p) + len(q) - 1)
    for i, pi in enumerate(p):
        for j, qj in enumerate(q):
            out[i + j] += pi * qj          # every pair contributes -> O(n^2)
    return out

# (1 + 2x)(3 + 4x) = 3 + 10x + 8x^2
print("(1+2x)(3+4x) ->", naive_conv([1, 2], [3, 4]))

# convolution is exactly what Python's % cross-multiplication of digits would give:
A, B = [random.randint(0, 9) for _ in range(6)], [random.randint(0, 9) for _ in range(6)]
print("A          :", A)
print("B          :", B)
print("conv(A, B) :", naive_conv(A, B))

(1+2x)(3+4x) -> [3, 10, 8]
A          : [6, 6, 0, 4, 8, 7]
B          : [6, 4, 7, 5, 9, 3]
conv(A, B) : [36, 60, 66, 96, 148, 174, 122, 125, 119, 87, 21]


## 2. FFT — recursive Cooley–Tukey + inverse, from scratch

The **discrete Fourier transform** evaluates a length-`n` coefficient vector at the `n` complex roots of unity `ω_n^k = e^{-2πik/n}`. The **Cooley–Tukey** FFT computes all n evaluations in O(n log n) by **divide & conquer**: split the coefficients into **even** and **odd** indices, recursively transform each half, then recombine with **twiddle factors**:

- `A(x) = A_even(x²) + x·A_odd(x²)`
- At `ω^k` and `ω^{k+n/2}` the two halves reuse the *same* sub-results (since `ω^{k+n/2} = -ω^k`), giving the **butterfly**: `out[k] = E[k] + t`, `out[k+n/2] = E[k] - t` with `t = ω^k·O[k]`.

That halving is why it's O(n log n) — the same recurrence as merge sort. It needs `n` to be a **power of two**. The **inverse FFT** is the same routine with conjugated roots, divided by `n` (we get it for free by conjugating in and out).

In [2]:
import cmath

def fft(a):
    """Recursive Cooley-Tukey DFT. len(a) must be a power of two."""
    n = len(a)
    if n == 1:
        return list(a)
    even = fft(a[0::2])                          # divide: even-indexed coeffs
    odd = fft(a[1::2])                           #         odd-indexed coeffs
    out = [0j] * n
    for k in range(n // 2):
        t = cmath.exp(-2j * cmath.pi * k / n) * odd[k]   # twiddle factor * odd half
        out[k] = even[k] + t                    # butterfly: w^k = -w^(k+n/2)
        out[k + n // 2] = even[k] - t
    return out

def ifft(a):
    """Inverse DFT: conjugate -> forward FFT -> conjugate -> divide by n."""
    n = len(a)
    y = fft([x.conjugate() for x in a])
    return [x.conjugate() / n for x in y]

# round-trip: ifft(fft(v)) == v
v = [1, 2, 3, 4]                                 # length 4 = 2^2
back = [round(x.real, 9) for x in ifft(fft(v))]
print("v             :", v)
print("ifft(fft(v))  :", back)
assert back == v
print("round-trip identity holds: OK")

v             : [1, 2, 3, 4]
ifft(fft(v))  : [1.0, 2.0, 3.0, 4.0]
round-trip identity holds: OK


### Convolution via FFT, and the proof it's correct

To convolve, pad both inputs to a common power-of-two length `≥ len(p)+len(q)-1`, FFT each, multiply the value vectors **pointwise** (O(n)), inverse-FFT, and round (the imaginary parts are float noise; integer inputs give integer outputs). The claim to prove: this equals the naive O(n²) convolution exactly.

In [3]:
def fft_conv(p, q):
    """O(n log n) convolution: evaluate -> pointwise multiply -> interpolate."""
    need = len(p) + len(q) - 1
    n = 1
    while n < need:
        n <<= 1                                 # next power of two
    fa = fft(p + [0] * (n - len(p)))            # evaluate A at the n-th roots of unity
    fb = fft(q + [0] * (n - len(q)))            # evaluate B
    fc = [x * y for x, y in zip(fa, fb)]        # pointwise product (= product polynomial's values)
    res = ifft(fc)                              # interpolate back to coefficients
    return [round(x.real) for x in res[:need]]  # round off float noise -> exact ints

random.seed(1)
# PROVE: FFT convolution == naive convolution over many random integer polynomials
for _ in range(200):
    n = random.choice([1, 2, 5, 8, 13, 32])
    p = [random.randint(-50, 50) for _ in range(n)]
    q = [random.randint(-50, 50) for _ in range(random.randint(1, 16))]
    assert fft_conv(p, q) == naive_conv(p, q)
print("fft_conv == naive_conv for 200 random integer polynomials: OK")

p, q = [3, 1, 4, 1, 5], [2, 7, 1, 8]
print("fft_conv :", fft_conv(p, q))
print("naive    :", naive_conv(p, q))

fft_conv == naive_conv for 200 random integer polynomials: OK
fft_conv : [6, 23, 18, 55, 29, 68, 13, 40]
naive    : [6, 23, 18, 55, 29, 68, 13, 40]


## 3. `numpy.fft` — the production tool

You almost never hand-roll the recursion above: `numpy.fft` ships a vectorized, mixed-radix FFT in C (it even handles non-power-of-two lengths). For **real** integer inputs the right pair is `rfft` / `irfft`, which exploit conjugate symmetry to do ~half the work. The recipe is identical — transform, multiply, inverse-transform, round — and it agrees with both our from-scratch FFT and the naive convolution.

In [4]:
import numpy as np

def np_conv(p, q):
    """Convolution via numpy's real FFT. The production-grade version of fft_conv."""
    need = len(p) + len(q) - 1
    n = 1
    while n < need:
        n <<= 1
    fc = np.fft.rfft(p, n) * np.fft.rfft(q, n)  # transform both, pointwise multiply
    res = np.fft.irfft(fc, n)[:need]
    return [round(x) for x in res]

random.seed(2)
# PROVE: numpy agrees with both our hand-rolled FFT and the naive O(n^2) result
for _ in range(200):
    p = [random.randint(0, 100) for _ in range(random.randint(1, 20))]
    q = [random.randint(0, 100) for _ in range(random.randint(1, 20))]
    ref = naive_conv(p, q)
    assert np_conv(p, q) == ref == fft_conv(p, q)
print("numpy.fft == fft_conv == naive_conv for 200 random cases: OK")

# numpy.convolve is the one-liner for full convolution (and uses this internally for large inputs):
p, q = [3, 1, 4, 1, 5], [2, 7, 1, 8]
print("np_conv          :", np_conv(p, q))
print("np.convolve      :", np.convolve(p, q).tolist())
print("naive_conv       :", naive_conv(p, q))

numpy.fft == fft_conv == naive_conv for 200 random cases: OK
np_conv          : [6, 23, 18, 55, 29, 68, 13, 40]
np.convolve      : [6, 23, 18, 55, 29, 68, 13, 40]
naive_conv       : [6, 23, 18, 55, 29, 68, 13, 40]


`numpy.convolve` is the everyday builtin for full convolution; for the *circular* convolution that FFT computes natively (and for 2-D image kernels) reach for `scipy.signal.fftconvolve`. numpy switches to an FFT-based method internally once the inputs are large, so you get the O(n log n) win without writing any of the above.

## 4. Applications — big-integer multiply, counting

**Big-integer multiplication.** Write each number in a base (here base 10, one digit per coefficient). The digit-vectors *convolve* to the product's pre-carry coefficients; a single carry pass normalizes them. This is exactly how arbitrary-precision libraries multiply huge numbers — CPython's own `int` uses schoolbook then Karatsuba, but GMP and competitive-programming kernels switch to FFT/NTT for very large operands.

**Counting via convolution.** If `cnt_a[v]` is how many ways value `v` appears in set A (an indicator/histogram), then `conv(cnt_a, cnt_b)[s]` counts the pairs `(x, y)` with `x + y = s` — e.g. the number of ways two dice sum to each total. Convolution *is* "combine every pair, bucket them by sum".

In [5]:
# --- big-integer multiplication via convolution + carry ---
def big_mul(x, y, base=10):
    """Multiply via digit convolution, then propagate carries."""
    dx = [int(c) for c in str(x)][::-1]         # least-significant digit first
    dy = [int(c) for c in str(y)][::-1]
    conv = fft_conv(dx, dy)                      # FFT convolution of the digit vectors
    out, carry = [], 0
    for v in conv:                              # normalize: each slot in [0, base)
        v += carry
        out.append(v % base)
        carry = v // base
    while carry:
        out.append(carry % base)
        carry //= base
    return int("".join(str(d) for d in reversed(out)) or "0")

random.seed(3)
for _ in range(100):                            # PROVE vs Python's own bignum multiply
    a = random.randint(0, 10 ** 12)
    b = random.randint(0, 10 ** 12)
    assert big_mul(a, b) == a * b
print("big_mul == Python's '*' for 100 random pairs: OK")
print("123456789 * 987654321 =", big_mul(123456789, 987654321), "==", 123456789 * 987654321)

big_mul == Python's '*' for 100 random pairs: OK
123456789 * 987654321 = 121932631112635269 == 121932631112635269


In [6]:
# --- counting: ways two dice sum to s = convolution of their face-histograms ---
die = [0, 1, 1, 1, 1, 1, 1]                     # index = face value; one way each for 1..6
sums = fft_conv(die, die)                       # sums[s] = # ordered pairs summing to s
for s in range(2, 13):
    print(f"  sum {s:>2}: {sums[s]} ways")

# brute-force cross-check
from collections import Counter
brute = Counter(i + j for i in range(1, 7) for j in range(1, 7))
assert all(sums[s] == brute[s] for s in range(2, 13))
print("convolution counts == brute-force pair counts: OK")

  sum  2: 1 ways
  sum  3: 2 ways
  sum  4: 3 ways
  sum  5: 4 ways
  sum  6: 5 ways
  sum  7: 6 ways
  sum  8: 5 ways
  sum  9: 4 ways
  sum 10: 3 ways
  sum 11: 2 ways
  sum 12: 1 ways
convolution counts == brute-force pair counts: OK


## 5. NTT — exact integer convolution over a prime field

Float FFT works, but for **large** integer convolutions the rounding eventually bites — once intermediate values exceed ~2⁵³ the double-precision mantissa can't round back correctly. The fix is the **Number-Theoretic Transform**: run the *same* butterfly algorithm in a **prime field** `ℤ/pℤ` instead of the complex numbers, replacing the root of unity `e^{-2πi/n}` with a **primitive n-th root of unity mod p**. All arithmetic is exact integer mod-p — zero float error.

The classic modulus is `p = 998244353 = 119·2²³ + 1`. Because `p−1` is divisible by a large power of two, it has roots of unity for every power-of-two length up to 2²³. Its **primitive root** is `g = 3`; then `g^{(p−1)/n}` is a primitive n-th root of unity mod p — the NTT's twiddle. We build an **iterative** NTT (bit-reversal + butterflies) so it doubles as a clean reference implementation.

In [7]:
MOD = 998244353          # = 119 * 2^23 + 1, NTT-friendly prime
G = 3                    # primitive root of MOD

def ntt(a, invert):
    """Iterative in-place NTT mod MOD. len(a) must be a power of two."""
    a = a[:]
    n = len(a)
    j = 0                                        # bit-reversal permutation
    for i in range(1, n):
        bit = n >> 1
        while j & bit:
            j ^= bit
            bit >>= 1
        j |= bit
        if i < j:
            a[i], a[j] = a[j], a[i]
    length = 2
    while length <= n:                           # butterfly over doubling block sizes
        w = pow(G, (MOD - 1) // length, MOD)     # primitive length-th root of unity mod p
        if invert:
            w = pow(w, MOD - 2, MOD)             # inverse root (Fermat inverse)
        for i in range(0, n, length):
            wn = 1
            for k in range(length // 2):
                u = a[i + k]
                v = a[i + k + length // 2] * wn % MOD
                a[i + k] = (u + v) % MOD
                a[i + k + length // 2] = (u - v) % MOD
                wn = wn * w % MOD
        length <<= 1
    if invert:
        ninv = pow(n, MOD - 2, MOD)              # divide by n in the field
        a = [x * ninv % MOD for x in a]
    return a

def ntt_conv(p, q):
    """Exact convolution mod MOD — no float error."""
    need = len(p) + len(q) - 1
    n = 1
    while n < need:
        n <<= 1
    fa = ntt(p + [0] * (n - len(p)), False)
    fb = ntt(q + [0] * (n - len(q)), False)
    fc = [x * y % MOD for x, y in zip(fa, fb)]   # pointwise product in the field
    return ntt(fc, True)[:need]

random.seed(4)
# PROVE: NTT convolution == naive convolution EXACTLY (results kept below MOD)
for _ in range(200):
    p = [random.randint(0, 1000) for _ in range(random.randint(1, 16))]
    q = [random.randint(0, 1000) for _ in range(random.randint(1, 16))]
    ref = naive_conv(p, q)
    assert max(ref) < MOD                        # no wraparound -> mod result is the true value
    assert ntt_conv(p, q) == ref
print("ntt_conv == naive_conv EXACTLY for 200 random cases: OK")
print("ntt_conv([3,1,4,1,5], [2,7,1,8]) =", ntt_conv([3, 1, 4, 1, 5], [2, 7, 1, 8]))

ntt_conv == naive_conv EXACTLY for 200 random cases: OK
ntt_conv([3,1,4,1,5], [2,7,1,8]) = [6, 23, 18, 55, 29, 68, 13, 40]


`g = 3` really is a primitive root: `pow(3, (MOD-1)//2, MOD)` equals `MOD-1` (= −1), so 3 has the full multiplicative order `MOD−1` and thus yields a root of unity for every power-of-two length up to 2²³. For values that *do* exceed MOD, run the NTT under several such primes and recombine with the **Chinese Remainder Theorem** (**number theory**) to recover the exact big result — the standard "multi-prime NTT" used in fast big-integer multiplication.

In [8]:
# g = 3 has full order MOD-1  ->  primitive root, valid twiddle for all 2^k lengths
print("pow(3, (MOD-1)//2, MOD) =", pow(3, (MOD - 1) // 2, MOD), "== MOD-1 =", MOD - 1)
assert pow(G, (MOD - 1) // 2, MOD) == MOD - 1

# and NTT agrees with the float FFT on inputs both can handle:
random.seed(5)
p = [random.randint(0, 9) for _ in range(8)]
q = [random.randint(0, 9) for _ in range(8)]
assert ntt_conv(p, q) == fft_conv(p, q) == naive_conv(p, q)
print("NTT == FFT == naive on a shared example: OK")

pow(3, (MOD-1)//2, MOD) = 998244352 == MOD-1 = 998244352
NTT == FFT == naive on a shared example: OK


## 6. When to use + cheat-sheet

**Reach for the builtin first.** For everyday convolution, `numpy.convolve` / `scipy.signal.fftconvolve` already pick the O(n log n) FFT path for large inputs — you rarely write a transform by hand. Hand-rolled FFT/NTT earns its keep in competitive programming and bignum kernels where you need exactness or a custom field.

| You need… | Reach for | Why |
|---|---|---|
| Convolve two arrays (general) | `numpy.convolve` | C-level, auto FFT for large n |
| 2-D / image / signal convolution | `scipy.signal.fftconvolve` | FFT-based, multi-dimensional |
| Forward / inverse DFT, spectra | `numpy.fft.fft` / `rfft` | vectorized, any length |
| **Exact** integer convolution | **NTT** (`ntt_conv`) | integer-only, zero float error |
| Float convolution, big but inexact OK | **FFT** (`fft_conv`) | simplest O(n log n) route |
| Tiny inputs (n ≲ 64) | naive double loop | low constant beats FFT setup |

**Method comparison**

| method | field | error | n restriction | use when |
|---|---|:---:|---|---|
| **naive** | any ring | none | none | small n; clarity |
| **FFT** | ℂ (floats) | rounding | pad to 2^k | float/inexact convolution |
| **NTT** | ℤ/pℤ (exact) | **none** | pad to 2^k, p NTT-friendly | exact integer convolution |

**Complexity cheat-sheet**

| operation | naive | FFT / NTT | note |
|---|:---:|:---:|---|
| Polynomial multiply / convolution | O(n²) | **O(n log n)** | the headline win |
| Forward / inverse transform | O(n²) | O(n log n) | Cooley–Tukey butterflies |
| Pointwise multiply (in value space) | — | O(n) | where the speedup is spent |
| Big-integer multiply (N digits) | O(N²) | O(N log N) | digits as coefficients + carry |

<sub>FFT/NTT require padding to a power of two (mixed-radix variants relax this). NTT needs an NTT-friendly prime such as 998244353 = 119·2²³+1 with primitive root 3. For exact results larger than one prime can hold, combine several primes via CRT (**number theory**).</sub>